# Preparing the data for model training and evaluation

We use the cleaned flow records from [Notebook 1](01_data_exploration.ipynb) to prepare the dataset for model training and evaluation.

## 1. Setup

In [1]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from sklearn.model_selection import train_test_split

from ids_ml.data import (
    IDENTIFIER_COLUMNS, LABEL_ORDER, RANDOM_STATE, TEST_SIZE,
    VALIDATION_SIZE_WITHIN_TRAINING, find_project_root, processed_dataset_path,
)


def show_table(frame):
    display({"text/html": frame.to_html(index=False),
             "text/plain": frame.to_string(index=False)}, raw=True)

## 2. Choosing the features

Not every feature is useful for recognizing attacks from network behavior. Some describe the connection itself, while others identify where or when it was recorded. In this section, we decide which features to keep for training. The `Label` column serves a different purpose: it gives the class we want the model to predict.

### 2.1 Dataset and target

We check the dimensions against the cleaned dataset from Notebook 1 and verify that all 15 labels are present, with no unlabeled rows.

In [2]:
project = find_project_root()
path = processed_dataset_path(project)
file = pq.ParquetFile(path)
schema = file.schema_arrow
labels = pd.read_parquet(path, columns=["Label"])["Label"]
assert (file.metadata.num_rows, len(schema)) == (2_824_745, 76), "Expected the revised Notebook 1 export."
assert schema.names.count("Label") == 1 and len(set(schema.names)) == len(schema)
assert labels.notna().all() and set(labels.unique()) == set(LABEL_ORDER)
original_counts = labels.value_counts().reindex(LABEL_ORDER)

show_table(pd.DataFrame({
    "Measure": ["Rows", "Columns", "Target classes", "Missing labels"],
    "Count": [len(labels), len(schema), len(original_counts), labels.isna().sum()],
}))

Measure,Count
Rows,2824745
Columns,76
Target classes,15
Missing labels,0


### 2.2 Columns and data types

In [3]:
columns = pd.DataFrame({"Column": schema.names, "Type": [str(field.type) for field in schema]})
text_columns = set(columns.loc[columns["Type"].isin(["string", "large_string"]), "Column"])
assert text_columns == {*IDENTIFIER_COLUMNS, "Label"}
assert columns.loc[~columns["Column"].isin(text_columns), "Type"].eq("double").all()

is_text = columns["Column"].isin(text_columns)
listing = pd.concat([columns.loc[is_text], columns.loc[~is_text]])
show_table(listing.groupby("Type", sort=False).size().reset_index(name="Columns"))
display({"text/html": "<details><summary>Column names by type</summary>"
         + listing.to_html(index=False) + "</details>",
         "text/plain": listing.to_string(index=False)}, raw=True)

Type,Columns
string,5
double,71


Column,Type
Flow ID,string
Src IP,string
Dst IP,string
Timestamp,string
Label,string
Src Port,double
Dst Port,double
Protocol,double
Flow Duration,double
Total Fwd Packet,double


The five text columns are `Flow ID`, `Src IP`, `Dst IP`, `Timestamp`, and `Label`. The remaining 71 are numeric. Their types match the cleaned export; their meanings determine which ones belong in the model.

### 2.3 Feature exclusions

We exclude the flow ID, both IP addresses, and the timestamp because we want the classifier to recognize traffic behavior rather than particular machines or scheduled attack times. We keep the ports and protocol because they describe how the connection was made rather than identify a particular host.

In [4]:
source_features = [name for name in schema.names if name not in [*IDENTIFIER_COLUMNS, "Label"]]
assert set(IDENTIFIER_COLUMNS).issubset(schema.names) and len(source_features) == 71

X = pd.read_parquet(path, columns=source_features)
assert X.index.equals(labels.index) and X.index.is_unique
print(f"{X.shape[1]} features; target: Label.")

71 features; target: Label.


## 3. Rows with identical feature values

Once the features have been chosen, two distinct records can look identical to a classifier. In this section, we examine how often this happens and what it means for training and evaluation.

### 3.1 Exact matches and their distribution by class

We group rows that match on all 71 features, then compare the labels within each group. Including the label in the initial comparison would hide cases where identical features were recorded with different labels. The table counts both the rows in repeated groups and the extra copies: if three rows match, all three belong to the group, but only two are extra.

In [5]:
def exact_pattern_ids(frame):
    """Route rows into small buckets, then compare every source value within each bucket."""
    routing = frame[["Flow Duration", "Src Port", "Dst Port"]] + 0.0  # Treat signed zeros alike.
    buckets = pd.util.hash_pandas_object(routing, index=False).to_numpy() % 64
    ids = np.empty(len(frame), dtype=np.int64)
    offset = 0

    for bucket in np.unique(buckets):
        rows = np.flatnonzero(buckets == bucket)
        group = frame.iloc[rows].groupby(list(frame.columns), sort=False, dropna=False)
        local_ids = group.ngroup().to_numpy()
        ids[rows] = local_ids + offset
        offset += int(local_ids.max()) + 1

    return pd.Series(ids, index=frame.index, name="Pattern")


pattern = exact_pattern_ids(X)
sizes = pattern.value_counts()
repeated = sizes[sizes > 1]
extra = pattern.duplicated(keep="first")

labels_per_pattern = labels.groupby(pattern, sort=False).nunique()
conflicts = int(labels_per_pattern.gt(1).sum())

audit = pd.Series({
    "Total rows": len(X),
    "Distinct feature combinations": len(sizes),
    "Groups with more than one row": len(repeated),
    "Rows in those groups": repeated.sum(),
    "Extra copies beyond the first row": extra.sum(),
    "Groups with different labels": conflicts,
})
show_table(audit.rename_axis("Measure").reset_index(name="Count"))
assert conflicts == 0, "Resolve conflicting labels before keeping one row per pattern."

Measure,Count
Total rows,2824745
Distinct feature combinations,2808026
Groups with more than one row,10046
Rows in those groups,26765
Extra copies beyond the first row,16719
Groups with different labels,0


The 10,046 repeated groups contain 26,765 rows. Keeping one row per group would remove 16,719 extra copies, or **0.592% of the 2,824,745 rows**. The labels agree within every group.

To see whether repetition is equally common across classes, we compare the number of extra copies with the total for each affected class.

In [6]:
impact = pd.DataFrame({
    "Original rows": original_counts,
    "Extra copies": labels[extra].value_counts().reindex(LABEL_ORDER, fill_value=0),
    "Largest group": pattern.map(sizes).groupby(labels).max(),
})
impact["% of class that is extra"] = (100 * impact["Extra copies"] / impact["Original rows"]).round(3)
impact["Distinct patterns"] = impact["Original rows"] - impact["Extra copies"]

affected = impact.loc[impact["Extra copies"].gt(0), ["Original rows", "Extra copies", "% of class that is extra", "Largest group"]]
show_table(affected.sort_values("Extra copies", ascending=False).rename_axis("Label").reset_index())

Label,Original rows,Extra copies,% of class that is extra,Largest group
BENIGN,2268384,9132,0.403,379
DoS Hulk,229964,7560,3.287,5
DoS slowloris,5796,27,0.466,5


Only BENIGN, DoS Hulk, and DoS slowloris contain extra copies; the other 12 classes have none. BENIGN has the largest repeated group: 379 rows share the same values across all 71 features. DoS Hulk has the highest proportion of extra copies (3.29%), while DoS slowloris has just 27 extra copies in total. In both attack classes, the largest repeated group contains five rows.

### 3.2 Keeping one row per feature combination

We want to compare how well models classify distinct examples of network traffic. If two models disagree on one example, the weight of that disagreement should not depend on whether its feature values occur twice or 379 times. We therefore keep one row per distinct combination of the 71 features. During training, repeated copies no longer give one example extra weight. During evaluation, a model must classify a different example correctly to earn another correct prediction.

In [7]:
X_model = X.loc[~extra]
y_model = labels.loc[X_model.index]
model_patterns = pattern.loc[X_model.index]

assert X_model.index.equals(y_model.index)
assert model_patterns.is_unique and len(model_patterns) == len(sizes)
assert len(X) - len(X_model) == int(impact["Extra copies"].sum())
assert y_model.value_counts().reindex(LABEL_ORDER).equals(impact["Distinct patterns"])

show_table(pd.DataFrame({
    "Measure": ["Rows before removal", "Extra copies removed", "Rows remaining"],
    "Count": [len(X), extra.sum(), len(X_model)],
}))
del X

Measure,Count
Rows before removal,2824745
Extra copies removed,16719
Rows remaining,2808026


## 4. Training, validation, and test split

We set aside 20% of the remaining rows for testing. Of the other 80%, one fifth goes to validation so that models can be compared without using the test set. This leaves 64% of all rows for training and 16% for validation. We stratify both splits by label to keep the class proportions close to those of the modeling data.

In [8]:
development, test = train_test_split(
    X_model.index.to_numpy(), test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_model,
)
training, validation = train_test_split(
    development, test_size=VALIDATION_SIZE_WITHIN_TRAINING,
    random_state=RANDOM_STATE, stratify=y_model.loc[development],
)
parts = {"Training": training, "Validation": validation, "Test": test}

assigned = np.concatenate(list(parts.values()))
assert np.array_equal(np.sort(assigned), np.sort(X_model.index.to_numpy()))

show_table(pd.DataFrame([
    {"Partition": name, "Rows": len(rows), "% of all rows": round(100 * len(rows) / len(X_model), 2)}
    for name, rows in parts.items()
]))

Partition,Rows,% of all rows
Training,1797136,64.0
Validation,449284,16.0
Test,561606,20.0


These percentages describe the dataset as a whole. For a class with very few examples, the actual counts matter more. We compare each class’s training, validation, and test counts with the number of rows it had before the split.

In [9]:
counts = pd.DataFrame({
    name: y_model.loc[rows].value_counts().reindex(LABEL_ORDER, fill_value=0)
    for name, rows in parts.items()
})
assert counts.gt(0).all().all()
assert counts.sum(axis=1).equals(impact["Distinct patterns"])

show_table(counts.assign(Total=impact["Distinct patterns"]).sort_values("Total").rename_axis("Label").reset_index())

Label,Training,Validation,Test,Total
Heartbleed,5,1,1,7
Web Attack - Sql Injection,14,3,4,21
Infiltration,22,6,7,35
Web Attack - XSS,418,104,130,652
Web Attack - Brute Force,964,241,302,1507
Bot,1252,313,391,1956
DoS Slowhttptest,3519,880,1100,5499
DoS slowloris,3692,923,1154,5769
SSH-Patator,3773,943,1179,5895
FTP-Patator,5076,1269,1586,7931


All 15 classes are represented. For Heartbleed, however, there are five training rows and only one row in each holdout. Correctly classifying that single row gives 100% recall for Heartbleed; missing it gives 0%. The split preserves the class, but it cannot give us a precise estimate of how often a model would recognize it.

## 5. Save the training, validation, and test data

We save the training and validation rows separately so that each experiment can use the same split. The test rows go into their own file for final evaluation.

In [10]:
filenames = {
    "Training": "cicids2017_train.parquet",
    "Validation": "cicids2017_validation.parquet",
    "Test": "cicids2017_test.parquet",
}

for name, rows in parts.items():
    target = path.parent / filenames[name]
    partition = X_model.loc[rows]
    partition["Label"] = y_model.loc[rows]
    partition.to_parquet(target, index=False)

    if not target.is_file():
        raise FileNotFoundError(f"The partition file was not created: {target.name}")
    del partition

print(f"Columns saved per file: {len(source_features) + 1} ({len(source_features)} features and Label)")
saved_files = [path.parent / filename for filename in filenames.values()]
show_table(pd.DataFrame([
    {"Saved file": target.relative_to(project).as_posix(), "Size (MiB)": round(target.stat().st_size / 1024**2, 2)}
    for target in saved_files
]))

Columns saved per file: 72 (71 features and Label)


Saved file,Size (MiB)
ml/data/processed/cicids2017_train.parquet,241.07
ml/data/processed/cicids2017_validation.parquet,60.35
ml/data/processed/cicids2017_test.parquet,75.11
